# Ruhrwatt gold: price forecast and load shifting

Needs `make gold` and `make backtest`. Plant presets are illustrative assumptions, savings cover the day-ahead energy
cost only (upper bound: no ramp limits, grid fees or intraday trading).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from ruhrwatt.gold import shift

if "RUHRWATT_DATA_DIR" in os.environ:
    DATA = Path(os.environ["RUHRWATT_DATA_DIR"]).expanduser()
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    DATA = ROOT / "data"

feats = pl.read_parquet(DATA / "gold" / "price_features_15min.parquet")
shift_days = pl.read_parquet(DATA / "gold" / "load_shift_daily.parquet")
monthly = pl.read_parquet(DATA / "gold" / "market_monthly.parquet")
qh = pl.read_parquet(DATA / "silver" / "energy_15min.parquet")
daily = pl.read_parquet(DATA / "silver" / "energy_daily.parquet")
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})


# Berlin wall-clock time for plotting (matplotlib would draw tz-aware times in UTC).
def local(df: pl.DataFrame):
    return df["ts_berlin"].dt.replace_time_zone(None)

## Model comparison (walk-forward)

Reads the results of `make backtest` (run it first; it takes a few minutes). Skill = how much
lower the MAE is than "same quarter-hour yesterday".

In [ ]:
G = DATA / "gold"
scores = pl.read_parquet(G / "backtest_scores.parquet")
by_hour = pl.read_parquet(G / "backtest_by_hour.parquet")
preds = pl.read_parquet(G / "backtest_predictions.parquet")
imp = pl.read_parquet(G / "feature_importance.parquet")
scores

In [ ]:
s = scores.sort("mae", descending=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(s["model"], s["mae"])
for y, v in enumerate(s["mae"]):
    ax.text(v, y, f" {v:.1f}", va="center")
ax.set_xlabel("MAE, EUR/MWh (lower is better)")
ax.set_title("Day-ahead price forecast error, walk-forward test");

In [ ]:
fig, ax = plt.subplots()
for m in ["naive_d1", "ridge", "lightgbm", "catboost", "lightgbm_no_fc"]:
    if m in by_hour.columns:
        ax.plot(by_hour["hour_berlin"], by_hour[m], marker="o", label=m)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("hour (Berlin)")
ax.set_ylabel("MAE, EUR/MWh")
ax.set_title("Where the forecast is hard: error by hour of day")
ax.legend();

In [ ]:
top = imp.head(15).sort("share")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["feature"], top["share"] * 100)
ax.set_xlabel("% of total gain (LightGBM, last fold)")
ax.set_title("Most important features");

In [ ]:
last = preds.filter(pl.col("date_berlin") >= preds["date_berlin"].max() - pl.duration(days=6))
t = local(last)
fig, ax = plt.subplots()
ax.fill_between(t, last["p10"], last["p90"], alpha=0.25, label="P10-P90")
ax.plot(t, last["price_eur_mwh"], lw=1.4, color="k", label="actual")
ax.plot(t, last["pred_lightgbm"], lw=1, label="LightGBM")
ax.plot(t, last["pred_naive_d1"], lw=1, alpha=0.6, label="naive: yesterday")
ax.set_ylabel("EUR/MWh")
ax.set_title("Last test week: forecast and P10-P90 range")
ax.legend();

## What load shifting looks like on one day

In [ ]:
plant = shift.PRESETS["chlor_alkali"]
best_day = shift_days.filter(pl.col("plant") == plant.name).sort(
    "savings_eur_per_mwh", descending=True
)["date_berlin"][0]
sched = shift.schedule(shift.priced_days(qh, daily), plant).filter(
    pl.col("date_berlin") == best_day
)
fig, ax = plt.subplots()
ax.step(local(sched), sched["flat_mwh"] * 4, where="post", label="flat (MW)")
ax.step(local(sched), sched["shifted_mwh"] * 4, where="post", label="shifted (MW)")
ax.set_ylabel("MW")
ax2 = ax.twinx()
ax2.plot(local(sched), sched["price_eur_mwh"], color="k", lw=1, alpha=0.6)
ax2.set_ylabel("EUR/MWh")
ax.set_title(f"{plant.label}: best day {best_day}")
ax.legend(loc="upper left");

## Savings per month and plant

In [ ]:
cols = [c for c in monthly.columns if c.startswith("shift_savings_eur_mwh_")]
fig, ax = plt.subplots()
for c in cols:
    ax.plot(
        monthly["month"], monthly[c], marker="o", label=c.removeprefix("shift_savings_eur_mwh_")
    )
ax.set_ylabel("EUR saved per MWh consumed")
ax.set_title("Within-day load shifting value by month")
ax.legend()
shift.summary(shift_days)

## Savings follow the daily price spread

In [ ]:
d = shift_days.filter(pl.col("plant") == "generic_10pct").join(
    daily.select("date_berlin", "price_spread"), on="date_berlin"
)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(d["price_spread"], d["savings_eur_per_mwh"], s=6, alpha=0.5)
ax.set_xlabel("daily price spread (max - min), EUR/MWh")
ax.set_ylabel("EUR saved per MWh")
ax.set_title("+/-10 % flexibility: value grows with the daily spread");